This notebook is to calculate SST and precipitation regressions against CANI and EANI during JJA and SON

# Imports

In [2]:
import xarray as xr
import numpy as np
from scipy.stats import t

## Indices

In [3]:
indices = xr.open_dataset("Results/CANI_EANI_CESM21.nc")
indices

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 1212, member: 100)
Coordinates:
    month    (time) int64 10kB ...
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
Data variables:
    EANI     (member, time) float64 970kB ...
    CANI     (member, time) float64 970kB ...
Attributes:
    title:        Atlantic Niño Index Timeseries
    description:  Contains EANI, CANI calculated from CESM2.1 LE2
    created:      2025-07-14

## Precipitation

In [4]:
precip_ds = xr.open_dataset('/glade/work/acruz/CESM21PRECT.nc')
precip_ds = precip_ds.sel(time=slice('1914-01-01', '2014-12-01'))
precip_ds.coords['lon'] = (precip_ds.coords['lon'] + 180) % 360 - 180
precip_ds = precip_ds.sortby(precip_ds.lon)
precip_ds = precip_ds.assign_coords({'member': precip_ds['member']})

In [5]:
precip_ds

<xarray.Dataset> Size: 27GB
Dimensions:  (lat: 192, time: 1212, member: 100, lon: 288)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
  * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
Data variables:
    PRECT    (member, time, lat, lon) float32 27GB ...

# function

In [6]:
def xr_regression(x, y, lag_x=0, lag_y=0, dim="time", alternative="two-sided"):
    """
    From https://stackoverflow.com/questions/52108417/how-to-apply-linear-regression-to-every-pixel-in-a-large-multi-dimensional-array
    requires scipy.stats as t
    Takes two xr.Datarrays of any dimensions (input data could be a 1D
    time series, or for example, have three dimensions e.g. time, lat,
    lon), and returns covariance, correlation, coefficient of
    determination, regression slope, intercept, p-value and standard
    error, and number of valid observations (n) between the two datasets
    along their aligned first dimension.

    Datasets can be provided in any order, but note that the regression
    slope and intercept will be calculated for y with respect to x.

    Inspired by:
    https://hrishichandanpurkar.blogspot.com/2017/09/vectorized-functions-for-correlation.html

    Parameters
    ----------
    x, y : xarray DataArray
        Two xarray DataArrays with any number of dimensions, both
        sharing the same first dimension
    lag_x, lag_y : int, optional
        Optional integers giving lag values to assign to either of the
        data, with lagx shifting x, and lagy shifting y with the
        specified lag amount.
    dim : str, optional
        An optional string giving the name of the dimension on which to
        align (and optionally lag) datasets. The default is 'time'.
    alternative : string, optional
        Defines the alternative hypothesis. Default is 'two-sided'.
        The following options are available:

        * 'two-sided': slope of the regression line is nonzero
        * 'less': slope of the regression line is less than zero
        * 'greater':  slope of the regression line is greater than zero

    Returns
    -------
    regression_ds : xarray.Dataset
        A dataset comparing the two input datasets along their aligned
        dimension, containing variables including covariance, correlation,
        coefficient of determination, regression slope, intercept,
        p-value and standard error, and number of valid observations (n).

    """

    # Shift x and y data if lags are specified
    if lag_x != 0:
        # If x lags y by 1, x must be shifted 1 step backwards. But as
        # the 'zero-th' value is nonexistant, xarray assigns it as
        # invalid (nan). Hence it needs to be dropped
        x = x.shift(**{dim: -lag_x}).dropna(dim=dim)

        # Next re-align the two datasets so that y adjusts to the
        # changed coordinates of x
        x, y = xr.align(x, y)

    if lag_y != 0:
        y = y.shift(**{dim: -lag_y}).dropna(dim=dim)

    # Ensure that the data are properly aligned to each other.
    x, y = xr.align(x, y)

    # Compute data length, mean and standard deviation along dim
    n = y.notnull().sum(dim=dim)
    xmean = x.mean(dim=dim)
    ymean = y.mean(dim=dim)
    xstd = x.std(dim=dim)
    ystd = y.std(dim=dim)

    # Compute covariance, correlation and coefficient of determination
    cov = ((x - xmean) * (y - ymean)).sum(dim=dim) / (n)
    cor = cov / (xstd * ystd)
    r2 = cor**2

    # Compute regression slope and intercept
    slope = cov / (xstd**2)
    intercept = ymean - xmean * slope

    # Compute t-statistics and standard error
    tstats = cor * np.sqrt(n - 2) / np.sqrt(1 - cor**2)
    stderr = slope / tstats

    # Calculate p-values for different alternative hypotheses.
    if alternative == "two-sided":
        pval = t.sf(np.abs(tstats), n - 2) * 2
    elif alternative == "greater":
        pval = t.sf(tstats, n - 2)
    elif alternative == "less":
        pval = t.cdf(np.abs(tstats), n - 2)

    # Wrap p-values into an xr.DataArray
    pval = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)

    # Combine into single dataset
    regression_ds = xr.merge(
        [
            cov.rename("cov").astype(np.float32),
            cor.rename("cor").astype(np.float32),
            r2.rename("r2").astype(np.float32),
            slope.rename("slope").astype(np.float32),
            intercept.rename("intercept").astype(np.float32),
            pval.rename("pvalue").astype(np.float32),
            stderr.rename("stderr").astype(np.float32),
            n.rename("n").astype(np.int16),
        ]
    )

    return regression_ds

# Data selection

In [7]:
jja_EANI = indices['EANI'].sel(time=indices['EANI'].time.dt.month.isin([6, 7, 8]))
jja_CANI = indices['CANI'].sel(time=indices['CANI'].time.dt.month.isin([6, 7, 8]))
jja_EANI

<xarray.DataArray 'EANI' (member: 100, time: 303)> Size: 242kB
[30300 values with dtype=float64]
Coordinates:
    month    (time) int64 2kB ...
  * time     (time) object 2kB 1914-06-01 00:00:00 ... 2014-08-01 00:00:00
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99

In [8]:
jja_precip = precip_ds['PRECT'].sel(time=precip_ds.time.dt.month.isin([6, 7, 8]))
son_precip = precip_ds['PRECT'].sel(time=precip_ds.time.dt.month.isin([9, 10, 11]))
jja_precip

<xarray.DataArray 'PRECT' (member: 100, time: 303, lat: 192, lon: 288)> Size: 7GB
[1675468800 values with dtype=float32]
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * time     (time) object 2kB 1914-06-01 00:00:00 ... 2014-08-01 00:00:00
  * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
Attributes:
    units:         m/s
    long_name:     Total (convective and large-scale) precipitation rate (liq...
    cell_methods:  time: mean

# Ntime dim

In [9]:
# fake coordinate to regress over different dates
ntime = np.arange(0, len(jja_precip['time']), 1)

jja_precip = jja_precip.assign_coords({'ntime': ('time', ntime)})
son_precip = son_precip.assign_coords({'ntime': ('time', ntime)})
jja_EANI = jja_EANI.assign_coords({'ntime': ('time', ntime)})
jja_CANI = jja_CANI.assign_coords({'ntime': ('time', ntime)})

jja_precip = jja_precip.swap_dims({'time': 'ntime'})
son_precip = son_precip.swap_dims({'time': 'ntime'})
jja_EANI = jja_EANI.swap_dims({'time': 'ntime'})
jja_CANI = jja_CANI.swap_dims({'time': 'ntime'})

# Regressions

## Test
testing was done on sst file

In [9]:
# x = jja_EANI.sel(member=slice(0, 1))
# y = jja_sst.sel(member=slice(0, 1))

In [10]:
# x

In [11]:
# y

In [12]:
# test = xr_regression(x, y, dim='ntime')
# test

## All members

In [14]:
jja_PRECTvEANI = xr_regression(jja_EANI, jja_precip, dim='ntime')
# save results due to long time calculation
jja_PRECTvEANI.to_netcdf('Results/CESM21/JJA_EANI_PRECT_reg.nc')

In [15]:
jja_PRECTvCANI = xr_regression(jja_CANI, jja_precip, dim='ntime')
jja_PRECTvCANI.to_netcdf('Results/CESM21/JJA_CANI_PRECT_reg.nc')

In [16]:
son_PRECTvEANI = xr_regression(jja_EANI, son_precip, dim='ntime')
son_PRECTvEANI.to_netcdf('Results/CESM21/SON_EANI_PRECT_reg.nc')

In [ ]:
son_PRECTvCANI = xr_regression(jja_CANI, son_precip, dim='ntime')
son_PRECTvCANI.to_netcdf('Results/CESM21/SON_CANI_PRECT_reg.nc')